# Урок 01 - Введение в AI-агентов

Добро пожаловать на первый урок курса **AI-агенты для начинающих**!

**AI-агент** — это программа, которая использует большую языковую модель (LLM) в качестве движка рассуждений и может выполнять *действия* в реальном мире — вызывать API, запрашивать базы данных или запускать код — чтобы выполнить задачу от имени пользователя.

В этом ноутбуке вы создадите своего первого агента: **Туристического агента**, который рекомендует места для отдыха. По ходу вы научитесь:

1. Подключаться к Microsoft Foundry Agent Service используя **Microsoft Agent Framework**.
2. Давать агенту **инструмент** — простую функцию на Python, которую он может вызывать.
3. Запускать агента и изучать его ответ.
4. Транслировать ответ агента по токенам.


## Настройка

Перед запуском этой тетради убедитесь, что у вас есть:

1. **Проект Microsoft Foundry** с развернутой моделью чата (например, `gpt-4.1-mini`).
2. **Вход в систему через Azure CLI** — выполните `az login` в вашем терминале.
3. **Установлены необходимые переменные окружения:**
   - `AZURE_AI_PROJECT_ENDPOINT` — конечная точка вашего проекта Microsoft Foundry.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — имя вашей развернутой модели.

В следующей ячейке устанавливаются необходимые вам пакеты Python.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Создание вашего первого агента

Агенту нужны две вещи:

- **Инструкции**, которые рассказывают ему, *кто он* и *как себя вести* (системный промпт).
- **Инструменты** — функции Python, украшенные `@tool`, которые агент может вызывать для получения информации или выполнения действий.

Ниже мы определяем простой инструмент, который возвращает список популярных туристических направлений. Агент будет использовать этот инструмент, когда пользователь попросит рекомендации по путешествиям.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Потоковые ответы

Для более интерактивного взаимодействия вы можете **транслировать** ответ агента. Вместо того чтобы ждать полного ответа, агент выдает текстовые фрагменты по мере их генерации. Это особенно полезно в чат-интерфейсах, где необходимо отображать результат в режиме реального времени.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Резюме

В этом уроке вы научились:

- **Создавать провайдер** для подключения к Microsoft Foundry Agent Service через `FoundryChatClient`.
- **Определять инструмент** с помощью декоратора `@tool`, чтобы агент мог вызывать ваши функции на Python.
- **Запускать агента** с сообщением пользователя и выводить его ответ.
- **Передавать ответы в потоке** для вывода в реальном времени.

В следующем уроке мы более подробно изучим агентские фреймворки и научимся давать агентам более мощные инструменты и возможности многоэтапного рассуждения.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от ответственности**:
Этот документ был переведен с использованием сервиса машинного перевода [Co-op Translator](https://github.com/Azure/co-op-translator). Несмотря на наши усилия по обеспечению точности, имейте в виду, что автоматический перевод может содержать ошибки или неточности. Оригинальный документ на его исходном языке следует считать авторитетным источником. Для получения критически важной информации рекомендуется обратиться к профессиональному человеческому переводу. Мы не несем ответственности за любые недоразумения или неправильные толкования, возникшие в результате использования этого перевода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
